# [LAB-13] 웹 데이터 수집하기

## 3. 웹 페이지 데이터 수집

### #01 준비작업

1. 라이브러리 참조

In [1]:
import os           # 파일, 폴더, 경로 처리
import sys          # 파이썬 실행 환경 관련 기능
import asyncio      # 비동기 작업 관리
from bs4 import BeautifulSoup       # HTML 파싱
from pandas import DataFrame        # 수집 결과를 표로 변환
from playwright.sync_api import sync_playwright         # 실제 브라우저를 조작해 웹페이지 접근

### #02 동적 웹 페이지로부터 데이터 수집

1. 크롬 브라우저 실행함수 정의

In [2]:
# 브라우저 실행 함수 정의 -> 접속할 웹 페이지 주소를 파라미터로 받는다.
def run(url):
    with sync_playwright() as p:
        # 브라우저 객체 생성
        # -> headless=False: 브라우저 창 띄우기
        #    (True: 창을 띄우지 않고 백그라운드 실행)
        # -> slow_mo=500: 각 동작 사이에 500s 지연 (속도 조절)
        browser = p.chromium.launch(headless=False, slow_mo=500)

        page = browser.new_page()       # 새 페이지 열기
        page.goto(url)                  # 페이지로 이동
        html = page.content()           # 웹 브라우저에 렌더링된 HTML 코드 가져오기
        browser.close()                 # 브라우저 닫기
        return html                     # 수집된 소스코드 리턴

2. 함수를 호출해 크롬 브라우저 실행하기

In [3]:
# Windows에서 asyncio 루프 정책 설정 (Playwright와 호환되도록)
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# run 함수를 윈도우 레벨에서 호출해 웹 페이지의 HTML 소스코드를 가져온다
html = await asyncio.to_thread(run, "https://data.hossam.kr/py/myfood/")

print(html)

<!DOCTYPE html><html lang="en"><head><meta charset="utf-8"><link rel="icon" href="/py/myfood/favicon.ico"><meta name="viewport" content="width=device-width,initial-scale=1"><meta name="theme-color" content="#000000"><meta name="description" content="Web site created using create-react-app"><link rel="apple-touch-icon" href="/py/myfood/logo192.png"><link rel="manifest" href="/py/myfood/manifest.json"><title>React App</title><link rel="preconnect" href="https://fonts.googleapis.com"><link rel="preconnect" href="https://fonts.gstatic.com" crossorigin=""><link href="https://fonts.googleapis.com/css2?family=Noto+Sans+KR:wght@100..900&amp;display=swap" rel="stylesheet"><link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/7.0.0/css/all.min.css"><script defer="defer" src="/py/myfood/static/js/main.3378ef15.js"></script><style data-styled="active" data-styled-version="6.1.19"></style></head><body><noscript>You need to enable JavaScript to run this app.</noscript><div

3. 웹 페이지 소스코드에서 원하는 데이터 추출

In [4]:
soup = BeautifulSoup(html)

food_item = soup.select(".food-item")       # 게시글 단위로 추출
resultset = []                              # 결과가 저장될 리스트


for item in food_item:
    title = item.select(".food-content h2")[0].text.strip()         # 게시글 하나의 단위에서 제목 추출
    desc = item.select(".food-content p")[0].text.strip()           # 게시글 하나의 단위에서 내용 추출

    img = item.select("img")                # 게시글 하나의 단위에서 이미지 태그 추출
    src = img[0].attrs["src"]               # 이미지 태그에서 이미지 주소 추출
    basename = os.path.basename(src)        # 이미지 주소에서 파일명만 가져옴

    resultset.append({"title": title, "desc": desc, "img": src})            # 빈 리스트에 데이터 추가

df = DataFrame(resultset)           # 데이터 프레임 생성
df

,title,desc,img
0,Food Item 1,"Just some random text, lorem ipsum text praese...",/py/myfood/static/media/cherries.7ef56d51c7c24...
1,Food Item 2,"Just some random text, lorem ipsum text praese...",/py/myfood/static/media/croissant.7d9a05f991cb...
2,Food Item 3,"Just some random text, lorem ipsum text praese...",/py/myfood/static/media/popsicle.8988496ab1464...
3,Food Item 4,"Just some random text, lorem ipsum text praese...",/py/myfood/static/media/salmon.b53f37e6551f124...
4,Food Item 5,"Just some random text, lorem ipsum text praese...",/py/myfood/static/media/sandwich.070d2a53863bb...
5,Food Item 6,"Just some random text, lorem ipsum text praese...",/py/myfood/static/media/steak.884245d345083526...
6,Food Item 7,"Just some random text, lorem ipsum text praese...",/py/myfood/static/media/steak2.a2fb608d8a681b5...
7,Food Item 8,"Just some random text, lorem ipsum text praese...",/py/myfood/static/media/wine.f385aa5175d23c02b...


## 연습문제 : 요일별 네이버 웹툰 데이터 수집

□ https://comic.naver.com/webtoon?tab={요일명} 페이지에 접속하여
월요일부터 일요일까지의 인기순 웹툰 목록에 대한 [요일명, 제목, 작가명, 평점]을 수집하세요.

◎ 수집된 결과는 하나의 csv 파일로 저장하면 됩니다.

◎ 요일명은 "mon", "tue", "wed", "thu", "fri", "sat", "sun" 중 하나입니다.

In [5]:
#import os           # 파일, 폴더, 경로 처리
#import sys          # 파이썬 실행 환경 관련 기능
#import asyncio      # 비동기 작업 관리
#from bs4 import BeautifulSoup       # HTML 파싱
#from pandas import DataFrame        # 수집 결과를 표로 변환
#from playwright.sync_api import sync_playwright         # 실제 브라우저를 조작해 웹페이지 접근

In [6]:
# 브라우저 실행 함수 정의 -> 접속할 웹 페이지 주소를 파라미터로 받는다.
def run(url):
    with sync_playwright() as p:
        # 브라우저 객체 생성
        # -> headless=False: 브라우저 창 띄우기
        #    (True: 창을 띄우지 않고 백그라운드 실행)
        # -> slow_mo=500: 각 동작 사이에 500s 지연 (속도 조절)
        browser = p.chromium.launch(headless=False, slow_mo=500)

        page = browser.new_page()       # 새 페이지 열기
        page.goto(url)                  # 페이지로 이동
        html = page.content()           # 웹 브라우저에 렌더링된 HTML 코드 가져오기
        browser.close()                 # 브라우저 닫기
        return html                     # 수집된 소스코드 리턴

In [7]:
# Windows에서 asyncio 루프 정책 설정 (Playwright와 호환되도록)
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

weekdays = ["mon", "tue", "wed", "thu", "fri", "sat", "sun"]
resultset = []                            # 결과가 저장될 리스트

for week in weekdays:
    # run 함수를 윈도우 레벨에서 호출해 웹 페이지의 HTML 소스코드를 가져온다
    html = await asyncio.to_thread(run, f"https://comic.naver.com/webtoon?tab={week}")

    soup = BeautifulSoup(html)

#   webtoon_item = soup.select(".item")
    webtoon_item = soup.select(".ContentList__info_area--bXx7h")       # 웹툰 단위로 추출


    for item in webtoon_item:
        title = item.select(".text")[0].text.strip()
        author = item.select(".ContentAuthor__author--CTAAP")[0].text.strip()
        rating = item.select(".rating_area .text")[0].text.strip()
#       title = item.select(".text")[0].text.strip()         # 웹툰 하나의 단위에서 제목 추출
#       author = item.select("[class^=\"ContentAuthor__author\"]")[0].text.strip()           # 웹툰 하나의 단위에서 작가명 추출
#       rating = item.select(".rating_area .text")[0].text.strip()

        resultset.append({"요일명": week, "제목": title, "작가명": author, "평점": rating})            # 빈 리스트에 데이터 추가

df = DataFrame(resultset)           # 데이터 프레임 생성
df

,요일명,제목,작가명,평점
0,mon,신체,엄세윤 / 정썸머,9.83
1,mon,참교육,채용택 / 한가람,9.90
2,mon,환생천마,JP / 부겸 / 장영훈,9.94
3,mon,샤MONEY즘,나락 / 영기,9.82
4,mon,시한부 천재가 살아남는 법,JP / 윤승기 / 청시소,9.83
...,...,...,...,...
801,sun,수사중 연애금지!,이재홍,9.88
802,sun,미스테리 쇼크,하랑,7.67
803,sun,챗윗미,조니조,9.90
804,sun,우리의 공백,조예빈,9.86


In [9]:
df.to_csv("요일별 네이버웹툰 데이터 수집.csv")